In [31]:
import sqlite3
import pandas as pd
import numpy as np
import re

# =====================================================
# 設定
# =====================================================

DB_PATH = "/Users/muna/Hana_research/data/db/Hana_Research.db"
OUTPUT_CSV = "/Users/muna/Desktop/ef_mentions_200patients.csv"

N_PATIENTS = 200
RANDOM_SEED = 42

# =====================================================
# DB接続
# =====================================================

conn = sqlite3.connect(DB_PATH)

# =====================================================
# EF関連患者抽出
# =====================================================

study_sql = """
SELECT DISTINCT Study_ID
FROM karte
WHERE Study_ID IS NOT NULL
AND (
       karte_text LIKE '%EF%'
    OR karte_text LIKE '%LVEF%'
    OR karte_text LIKE '%HFpEF%'
    OR karte_text LIKE '%HFPEF%'
    OR karte_text LIKE '%HFmrEF%'
    OR karte_text LIKE '%HFrEF%'
    OR karte_text LIKE '%preserved EF%'
    OR karte_text LIKE '%reduced EF%'
    OR karte_text LIKE '%左室駆出率%'
    OR karte_text LIKE '%駆出率%'
    OR karte_text LIKE '%収縮能低下%'
    OR karte_text LIKE '%左室収縮能低下%'
    OR karte_text LIKE '%左室機能低下%'
    OR karte_text LIKE '%systolic dysfunction%'
)
"""

study_df = pd.read_sql(study_sql, conn)

print(f"対象患者数: {len(study_df):,}")

# =====================================================
# ランダム200患者抽出
# =====================================================

sample_ids = (
    study_df["Study_ID"]
    .sample(
        n=min(N_PATIENTS, len(study_df)),
        random_state=RANDOM_SEED
    )
    .tolist()
)

print(f"抽出患者数: {len(sample_ids)}")

# =====================================================
# カルテ取得
# =====================================================

placeholders = ",".join(["?"] * len(sample_ids))

karte_sql = f"""
SELECT
    record_no,
    Study_ID,
    visit_datetime,
    karte_text
FROM karte
WHERE Study_ID IN ({placeholders})
"""

df = pd.read_sql(
    karte_sql,
    conn,
    params=sample_ids
)

print(f"対象カルテ数: {len(df):,}")

# =====================================================
# 正規表現
# =====================================================

range_patterns = [
    r'(?:EF|LVEF|左室駆出率|駆出率)\s*([0-9]{1,2})\s*[-－〜～]\s*([0-9]{1,2})\s*[%％]?',
]

lt_patterns = [
    r'(?:EF|LVEF|左室駆出率|駆出率)\s*[＜<]\s*([0-9]{1,2})'
]

gt_patterns = [
    r'(?:EF|LVEF|左室駆出率|駆出率)\s*[＞>]\s*([0-9]{1,2})'
]

single_patterns = [
    r'(?:EF|LVEF|左室駆出率|駆出率)\s*([0-9]{1,2})\s*[%％]?\s*(?:前後|程度|くらい|ぐらい)?'
]

# =====================================================
# 分類語
# =====================================================

classification_patterns = [
    ("HFpEF", r'HFpEF'),
    ("HFPEF", r'HFPEF'),
    ("HFmrEF", r'HFmrEF'),
    ("HFrEF", r'HFrEF'),

    ("preserved EF", r'preserved\s+EF'),
    ("reduced EF", r'reduced\s+EF'),

    ("EF正常", r'EF正常'),
    ("EF低下", r'EF低下'),

    ("左室収縮能低下", r'左室収縮能低下'),
    ("左室機能低下", r'左室機能低下'),
    ("収縮能低下", r'収縮能低下'),

    ("systolic dysfunction", r'systolic dysfunction'),
]

# =====================================================
# 抽出
# =====================================================

records = []

for _, row in df.iterrows():

    text = str(row["karte_text"])

    record_no = row["record_no"]
    study_id = row["Study_ID"]
    visit_datetime = row["visit_datetime"]

    # -----------------------------------------
    # EF range
    # -----------------------------------------

    for pattern in range_patterns:

        for m in re.finditer(pattern, text, flags=re.I):

            low = float(m.group(1))
            high = float(m.group(2))

            records.append({
                "Study_ID": study_id,
                "visit_datetime": visit_datetime,
                "record_no": record_no,
                "term_type": "EF_RANGE",
                "value_low": low,
                "value_high": high,
                "operator": None,
                "matched_text": m.group(0)
            })

    # -----------------------------------------
    # EF <
    # -----------------------------------------

    for pattern in lt_patterns:

        for m in re.finditer(pattern, text, flags=re.I):

            records.append({
                "Study_ID": study_id,
                "visit_datetime": visit_datetime,
                "record_no": record_no,
                "term_type": "EF",
                "value_low": float(m.group(1)),
                "value_high": None,
                "operator": "<",
                "matched_text": m.group(0)
            })

    # -----------------------------------------
    # EF >
    # -----------------------------------------

    for pattern in gt_patterns:

        for m in re.finditer(pattern, text, flags=re.I):

            records.append({
                "Study_ID": study_id,
                "visit_datetime": visit_datetime,
                "record_no": record_no,
                "term_type": "EF",
                "value_low": float(m.group(1)),
                "value_high": None,
                "operator": ">",
                "matched_text": m.group(0)
            })

    # -----------------------------------------
    # EF single
    # -----------------------------------------

    for pattern in single_patterns:

        for m in re.finditer(pattern, text, flags=re.I):

            records.append({
                "Study_ID": study_id,
                "visit_datetime": visit_datetime,
                "record_no": record_no,
                "term_type": "EF",
                "value_low": float(m.group(1)),
                "value_high": None,
                "operator": "=",
                "matched_text": m.group(0)
            })

    # -----------------------------------------
    # 分類
    # -----------------------------------------

    for term_type, pattern in classification_patterns:

        matches = set(
            m.group(0)
            for m in re.finditer(pattern, text, flags=re.I)
        )

        for hit in matches:

            records.append({
                "Study_ID": study_id,
                "visit_datetime": visit_datetime,
                "record_no": record_no,
                "term_type": term_type,
                "value_low": None,
                "value_high": None,
                "operator": None,
                "matched_text": hit
            })

# =====================================================
# DataFrame化
# =====================================================

ef_mentions = pd.DataFrame(records)

print(f"抽出件数: {len(ef_mentions):,}")

# =====================================================
# ソート
# =====================================================

ef_mentions = ef_mentions.sort_values(
    ["Study_ID", "visit_datetime"]
).reset_index(drop=True)

# =====================================================
# 保存
# =====================================================

ef_mentions.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

print()
print("保存完了")
print(OUTPUT_CSV)
print()
print(ef_mentions.head(30))

対象患者数: 2,450
抽出患者数: 200
対象カルテ数: 16,433
抽出件数: 6,566

保存完了
/Users/muna/Desktop/ef_mentions_200patients.csv

   Study_ID       visit_datetime record_no term_type  value_low  value_high  \
0   P000047   2015/6/25(木) 12:31      4541        EF       40.0         NaN   
1   P000047   2015/7/26(日) 13:20      4260        EF       50.0         NaN   
2   P000048   2016/4/27(水) 10:50     12720        EF       50.0         NaN   
3   P000122   2015/11/4(水) 19:36      2890        EF       40.0         NaN   
4   P000122  2016/10/19(水) 14:02      7053        EF       60.0         NaN   
5   P000122   2016/10/4(火) 13:50      7711        EF       60.0         NaN   
6   P000122  2016/11/16(水) 09:35      5703        EF       60.0         NaN   
7   P000122   2016/11/2(水) 14:42      6301        EF       60.0         NaN   
8   P000122  2016/12/21(水) 15:18      4277        EF       60.0         NaN   
9   P000122   2016/12/7(水) 13:40      4824        EF       60.0         NaN   
10  P000122   2016/9/21(水

In [32]:
ef_mentions[
    ef_mentions["term_type"]=="EF_RANGE"
].head(50)

,Study_ID,visit_datetime,record_no,term_type,value_low,value_high,operator,matched_text
87,P000261,2016/5/10(火) 10:30,12467,EF_RANGE,40.0,50.0,NaN,EF 40-50％
255,P000504,2017/1/4(水) 15:31,3836,EF_RANGE,55.0,60.0,NaN,EF 55-60％
594,P001558,2020/7/10(金) 14:00,1979,EF_RANGE,30.0,40.0,NaN,EF 30-40％
1228,P001702,2019/10/11(金) 09:07,1645,EF_RANGE,32.0,35.0,NaN,EF 32-35％
1233,P001702,2019/11/22(金) 09:55,10968,EF_RANGE,32.0,35.0,NaN,EF 32-35％
1238,P001702,2019/12/20(金) 09:29,8454,EF_RANGE,32.0,35.0,NaN,EF 32-35％
1249,P001702,2020/11/13(金) 09:02,1990,EF_RANGE,32.0,35.0,NaN,EF 32-35％
1256,P001702,2020/2/7(金) 09:55,4411,EF_RANGE,32.0,35.0,NaN,EF 32-35％
1261,P001702,2020/4/10(金) 09:16,10086,EF_RANGE,32.0,35.0,NaN,EF 32-35％
1266,P001702,2020/6/19(金) 09:02,4053,EF_RANGE,32.0,35.0,NaN,EF 32-35％


In [33]:
import sqlite3

conn = sqlite3.connect("/Users/muna/Hana_research/data/db/Hana_Research.db")
cursor = conn.cursor()

# SQLを実行
cursor.execute("SELECT * FROM last_one_month_diag LIMIT 10")

# 結果を取得
rows = cursor.fetchall()
for row in rows:
    print(row)

conn.close()

(160066, '老齢による筋力低下および廃用症候群', '【分類？】', 0)
(160066, '高血圧、高血圧性心疾患、腹部大動脈瘤', '分類不能、【心疾患】、【血管疾患】', 0)
(160066, '閉塞性肺疾患、左気胸手術後、小児肺結核既往', '【肺疾患】、分類不能', 0)
(160066, '左肋骨骨折疑い', '【整形疾患】', 0)
(160066, '過敏性腸症候群', '分類不能', 0)
(160066, '胃癌術後（2/3摘出）', '【癌】', 0)
(150148, '左下肺肺癌、癌性胸膜炎', '【癌】、【肺疾患】', 0)
(150148, 'アルツハイマー型認知症疑い、廃用症候群、嚥下障害', '【認知症】、【分類？】、分類不能', 0)
(150148, '高血圧症', '【分類？】', 0)
(150148, '左大転子部位褥瘡', '【褥瘡および皮膚疾患】', 0)


In [34]:
SELECT item, COUNT(*)
FROM ef_long
GROUP BY item;

SyntaxError: Invalid star expression (3967643235.py, line 1)

In [35]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("/Users/muna/Hana_research/data/db/Hana_Research.db")

df = pd.read_sql("""
    SELECT item, COUNT(*) as cnt
    FROM ef_long
    GROUP BY item
""", conn)

conn.close()

print(df)

               item    cnt
0             EF_GT    174
1             EF_LT     51
2           EF_high    355
3            EF_low    355
4            EF_mid    355
5          EF_value  40523
6          HF_class  38656
7  HF_class_numeric  40878
8           HF_term  38656


In [37]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("/Users/muna/Hana_research/data/db/Hana_Research.db")

df = pd.read_sql("""
    SELECT *
    FROM ef_long
    WHERE conflict_flag = 1
    LIMIT 100
""", conn)

conn.close()

print(df)

      id Study_ID      visit_datetime  visit_date record_no              item  \
0    338  P000067  2015/9/20(日) 10:12  2015-09-20      3554          EF_value   
1    339  P000067  2015/9/20(日) 10:12  2015-09-20      3554  HF_class_numeric   
2    340  P000067  2015/9/20(日) 10:12  2015-09-20      3554           HF_term   
3    341  P000067  2015/9/20(日) 10:12  2015-09-20      3554          HF_class   
4    416  P000067   2015/8/7(金) 11:10  2015-08-07      4130          EF_value   
..   ...      ...                 ...         ...       ...               ...   
95  2536  P000701  2018/2/27(火) 18:15  2018-02-27      1817          HF_class   
96  2544  P000701  2018/2/23(金) 11:16  2018-02-23      2026          EF_value   
97  2545  P000701  2018/2/23(金) 11:16  2018-02-23      2026  HF_class_numeric   
98  2546  P000701  2018/2/23(金) 11:16  2018-02-23      2026           HF_term   
99  2547  P000701  2018/2/23(金) 11:16  2018-02-23      2026          HF_class   

           value  matched_t

In [38]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(db_path)

pd.read_sql("""
SELECT
    conflict_flag,
    COUNT(*) AS n
FROM ef_long
GROUP BY conflict_flag
""", conn)

,conflict_flag,n
0,0,156603
1,1,3400


In [39]:
pd.read_sql("""
SELECT
    Study_ID,
    visit_datetime,
    COUNT(DISTINCT HF_class) as n_class
FROM (
    SELECT
        Study_ID,
        visit_datetime,
        value as HF_class
    FROM ef_long
    WHERE item='HF_class'
)
GROUP BY
    Study_ID,
    visit_datetime
HAVING n_class > 1
LIMIT 50
""", conn)

,Study_ID,visit_datetime,n_class
0,P000967,2021/10/14(木) 09:30,2
1,P000967,2021/10/28(木) 09:47,2
2,P000967,2021/11/11(木) 09:56,2
3,P000967,2021/11/25(木) 09:45,2
4,P000967,2021/12/23(木) 14:05,2
5,P000967,2021/12/9(木) 13:56,2
6,P000967,2022/1/20(木) 09:50,2
7,P000967,2022/1/6(木) 09:30,2
8,P000967,2022/10/20(木) 09:40,2
9,P000967,2022/10/6(木) 09:35,2


In [41]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("/Users/muna/Hana_research/data/db/Hana_Research.db")

df = pd.read_sql("""
    SELECT COUNT(DISTINCT Study_ID) as cnt
    FROM ef_long
    WHERE conflict_flag = 1
""", conn)

conn.close()

print(df)

   cnt
0   88


In [42]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("/Users/muna/Hana_research/data/db/Hana_Research.db")

df = pd.read_sql("""
    SELECT
        value,
        COUNT(*) as cnt
    FROM ef_long
    WHERE item = 'HF_term'
    GROUP BY value
    ORDER BY COUNT(*) DESC
""", conn)

conn.close()

print(df)

           value   cnt
0          hfpef  7875
1          HFpEF  7875
2          HFPEF  7875
3          hfref  2622
4          HFrEF  2622
5          HFREF  2622
6         hfmref  1917
7         HFmrEF  1917
8         HFMREF  1917
9          収縮能低下   553
10       左室収縮能低下   414
11  preserved EF   212
12          EF低下   204
13          EF正常    19
14        左室機能低下    11
15     normal EF     1


In [46]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("/Users/muna/Hana_research/data/db/Hana_Research.db")

pd.read_sql("""
SELECT item, COUNT(*) as cnt
FROM ef_long
GROUP BY item
ORDER BY item
""", conn)

,item,cnt
0,EF_value,50354
1,HF_class,8025
2,HF_class_numeric,50354
3,HF_term,8025


In [47]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("/Users/muna/Hana_research/data/db/Hana_Research.db")

pd.read_sql("""
SELECT item, COUNT(*) as cnt
FROM ef_long
GROUP BY item
ORDER BY item
""", conn)

,item,cnt
0,EF_value,50354
1,HF_class,8025
2,HF_class_numeric,50354
3,HF_term,8025


In [48]:
pd.read_sql("""
SELECT final_hf_class, COUNT(*) as cnt
FROM ef_long
GROUP BY final_hf_class
ORDER BY cnt DESC
""", conn)

,final_hf_class,cnt
0,HFpEF,81642
1,NaN,15762
2,HFrEF,11662
3,HFmrEF,6808
4,HFrEF_like,884


In [49]:
pd.read_sql("""
SELECT
    conflict_flag,
    COUNT(*) as cnt
FROM ef_long
GROUP BY conflict_flag
""", conn)

,conflict_flag,cnt
0,0,114114
1,1,2644


In [50]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("/Users/muna/Hana_research/data/db/Hana_Research.db")

pd.read_sql("""
SELECT
    hf_term_std,
    COUNT(*) AS n
FROM ef_long
WHERE hf_term_std IS NOT NULL
GROUP BY hf_term_std
ORDER BY n DESC
""", conn)

,hf_term_std,n
0,HFpEF,9158
1,HFrEF,3438
2,HFrEF_like,1774
3,HFmrEF,1680


In [51]:
pd.read_sql("""
SELECT
    hf_term_raw,
    hf_term_std,
    COUNT(*) AS n
FROM ef_long
WHERE hf_term_std='HFpEF'
GROUP BY hf_term_raw, hf_term_std
ORDER BY n DESC
""", conn)

,hf_term_raw,hf_term_std,n
0,HFpEF,HFpEF,8558
1,preserved EF,HFpEF,424
2,EF正常,HFpEF,88
3,HFPEF,HFpEF,86
4,normal EF,HFpEF,2


In [52]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("/Users/muna/Hana_research/data/db/Hana_Research.db")

df = pd.read_sql("""
    SELECT
        final_hf_class,
        COUNT(*) as cnt
    FROM (
        SELECT
            Study_ID,
            final_hf_class,
            MIN(visit_date) as first_visit
        FROM ef_long
        GROUP BY Study_ID
    )
    GROUP BY final_hf_class
    ORDER BY cnt DESC
""", conn)

conn.close()

print(df)

  final_hf_class   cnt
0          HFpEF  1903
1          HFrEF   247
2         HFmrEF   118
3     HFrEF_like     8
4            NaN     1


In [59]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("/Users/muna/Hana_research/data/db/Hana_Research.db")

df = pd.read_sql("""
    SELECT
        final_hf_class,
        COUNT(DISTINCT ef_long.Study_ID) as 患者数
    FROM ef_long
    WHERE ef_long.Study_ID IN (
        SELECT DISTINCT l.Study_ID
        FROM last_one_month_diag d
        JOIN study_id_linkage l ON d.Patient_ID = l.Patient_ID
        WHERE d.heart_failure_flg = 1
    )
    AND visit_date = (
        SELECT MIN(visit_date)
        FROM ef_long AS sub
        WHERE sub.Study_ID = ef_long.Study_ID
    )
    GROUP BY final_hf_class
    ORDER BY 患者数 DESC
""", conn)

conn.close()

print(df)
print(f"\n総数: {df['患者数'].sum()}")

  final_hf_class  患者数
0          HFpEF  632
1          HFrEF  188
2         HFmrEF   73
3     HFrEF_like    2

総数: 895


In [60]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("/Users/muna/Hana_research/data/db/Hana_Research.db")

# 1. heart_failure_flg=1の総患者数
df1 = pd.read_sql("""
    SELECT COUNT(DISTINCT Patient_ID) as heart_failure患者数
    FROM last_one_month_diag
    WHERE heart_failure_flg = 1
""", conn)
print("1. heart_failure_flg=1の総患者数")
print(df1)

# 2. study_id_linkageで紐付けできた患者数
df2 = pd.read_sql("""
    SELECT COUNT(DISTINCT l.Study_ID) as 紐付け済み患者数
    FROM last_one_month_diag d
    JOIN study_id_linkage l ON d.Patient_ID = l.Patient_ID
    WHERE d.heart_failure_flg = 1
""", conn)
print("\n2. study_id_linkageで紐付けできた患者数")
print(df2)

# 3. ef_longにデータがある患者数
df3 = pd.read_sql("""
    SELECT COUNT(DISTINCT ef_long.Study_ID) as ef_longあり患者数
    FROM ef_long
    WHERE ef_long.Study_ID IN (
        SELECT DISTINCT l.Study_ID
        FROM last_one_month_diag d
        JOIN study_id_linkage l ON d.Patient_ID = l.Patient_ID
        WHERE d.heart_failure_flg = 1
    )
""", conn)
print("\n3. ef_longにデータがある患者数")
print(df3)

conn.close()

1. heart_failure_flg=1の総患者数
   heart_failure患者数
0              1175

2. study_id_linkageで紐付けできた患者数
   紐付け済み患者数
0      1099

3. ef_longにデータがある患者数
   ef_longあり患者数
0           894
